## Packages

In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

## 1- Loading Data

In [2]:
train_data = pd.read_csv('/kaggle/input/solar-production-and-load-prediction-competition/train_data.csv')
test_data = pd.read_csv('/kaggle/input/solar-production-and-load-prediction-competition/test_data_masked.csv')
system_info = pd.read_csv('/kaggle/input/solar-production-and-load-prediction-competition/systems_new.csv')

## 2- Data Pre-processing

Changing the `timestamp` format for better visuals

In [3]:
train_data['timestamp'] = pd.to_datetime(train_data['timestamp'], format='%Y-%m-%d %H:%M:%S')
test_data['timestamp'] = pd.to_datetime(test_data['timestamp'], format='%Y-%m-%d %H:%M:%S')

In [4]:
train_data.head()

,system_id,timestamp,generation_W,load_W
0,17,2023-08-14 01:50:00,0.0,525.66667
1,17,2023-08-14 02:00:00,0.0,518.90000
2,17,2023-08-14 02:10:00,0.0,528.50001
3,17,2023-08-14 02:20:00,0.0,517.99999
4,17,2023-08-14 02:30:00,0.0,520.90001


In [5]:
system_info.head()

,system_id,connection_type,location,panels_capacity,load_capacity
0,1,RESIDENTIAL,ISLAMABAD,11.125,10.0
1,2,COMMERCIAL,SHEIKHUPURA,5.340,5.5
2,3,RESIDENTIAL,KARACHI,10.350,10.0
3,4,RESIDENTIAL,LAHORE,12.420,10.0
4,5,COMMERCIAL,KARACHI,30.295,20.0


Merging the `train` and `test` data with `system_new` file on system_id

In [6]:
train_data = pd.merge(train_data, system_info, on='system_id', how='left')
test_data = pd.merge(test_data, system_info, on='system_id', how='left')

In [7]:
train_data.head()

,system_id,timestamp,generation_W,load_W,connection_type,location,panels_capacity,load_capacity
0,17,2023-08-14 01:50:00,0.0,525.66667,COMMERCIAL,MARDAN,5.005,5.5
1,17,2023-08-14 02:00:00,0.0,518.90000,COMMERCIAL,MARDAN,5.005,5.5
2,17,2023-08-14 02:10:00,0.0,528.50001,COMMERCIAL,MARDAN,5.005,5.5
3,17,2023-08-14 02:20:00,0.0,517.99999,COMMERCIAL,MARDAN,5.005,5.5
4,17,2023-08-14 02:30:00,0.0,520.90001,COMMERCIAL,MARDAN,5.005,5.5


Splitting the data and time format in different cloumn for input

In [8]:
def create_time_features(df):
    df['hour'] = df['timestamp'].dt.hour
    df['day'] = df['timestamp'].dt.day
    df['month'] = df['timestamp'].dt.month
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    return df

train_data = create_time_features(train_data)
test_data = create_time_features(test_data)

In [9]:
train_data.head()

,system_id,timestamp,generation_W,load_W,connection_type,location,panels_capacity,load_capacity,hour,day,month,day_of_week
0,17,2023-08-14 01:50:00,0.0,525.66667,COMMERCIAL,MARDAN,5.005,5.5,1,14,8,0
1,17,2023-08-14 02:00:00,0.0,518.90000,COMMERCIAL,MARDAN,5.005,5.5,2,14,8,0
2,17,2023-08-14 02:10:00,0.0,528.50001,COMMERCIAL,MARDAN,5.005,5.5,2,14,8,0
3,17,2023-08-14 02:20:00,0.0,517.99999,COMMERCIAL,MARDAN,5.005,5.5,2,14,8,0
4,17,2023-08-14 02:30:00,0.0,520.90001,COMMERCIAL,MARDAN,5.005,5.5,2,14,8,0


ONe-Hot Encoding

In [10]:
train_data = pd.get_dummies(train_data, columns=['connection_type'])
test_data = pd.get_dummies(test_data, columns=['connection_type'])

In [11]:
train_data.head()

,system_id,timestamp,generation_W,load_W,location,panels_capacity,load_capacity,hour,day,month,day_of_week,connection_type_COMMERCIAL,connection_type_RESIDENTIAL
0,17,2023-08-14 01:50:00,0.0,525.66667,MARDAN,5.005,5.5,1,14,8,0,True,False
1,17,2023-08-14 02:00:00,0.0,518.90000,MARDAN,5.005,5.5,2,14,8,0,True,False
2,17,2023-08-14 02:10:00,0.0,528.50001,MARDAN,5.005,5.5,2,14,8,0,True,False
3,17,2023-08-14 02:20:00,0.0,517.99999,MARDAN,5.005,5.5,2,14,8,0,True,False
4,17,2023-08-14 02:30:00,0.0,520.90001,MARDAN,5.005,5.5,2,14,8,0,True,False


## 4- Feature Engineering

In [12]:
X_train = train_data.drop(columns=['system_id', 'generation_W', 'load_W', 'timestamp', 'location'])
y_train = train_data[['generation_W', 'load_W']]

X_test = test_data.drop(columns=['test_id', 'system_id', 'generation_W', 'load_W', 'timestamp', 'location'])

In [13]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [14]:
X_train.head()

,panels_capacity,load_capacity,hour,day,month,day_of_week,connection_type_COMMERCIAL,connection_type_RESIDENTIAL
1255951,14.850,20.0,3,9,9,5,False,True
2347909,11.700,10.0,16,24,2,5,False,True
2433567,5.400,5.5,7,24,11,4,False,True
3509432,11.375,10.0,16,11,7,3,True,False
983674,5.400,5.5,19,26,8,5,False,True


## 5- Model Training

In [15]:
my_xgb = xgb.XGBRegressor(
    objective='reg:absoluteerror',  # This specifies the regression task
    n_estimators=100,             # Number of trees
    learning_rate=0.05,            # Step size shrinkage used to prevent overfitting
    max_depth=7,                   # Maximum depth of a tree
    subsample=0.8,                 # Subsample ratio of the training instance
    colsample_bytree=0.8,          # Subsample ratio of columns when constructing each tree
    random_state=42,
)

In [16]:
my_xgb.fit(X_train, y_train, 
            eval_set=[(X_valid, y_valid)], 
            verbose=True)

[0]	validation_0-mae:1480.91149
[1]	validation_0-mae:1433.81242
[2]	validation_0-mae:1392.82138
[3]	validation_0-mae:1355.29913
[4]	validation_0-mae:1340.97000
[5]	validation_0-mae:1330.73619
[6]	validation_0-mae:1323.66030
[7]	validation_0-mae:1293.41857
[8]	validation_0-mae:1260.36494
[9]	validation_0-mae:1227.77206
[10]	validation_0-mae:1200.12237
[11]	validation_0-mae:1190.38631
[12]	validation_0-mae:1182.68962
[13]	validation_0-mae:1154.73253
[14]	validation_0-mae:1133.78614
[15]	validation_0-mae:1111.37329
[16]	validation_0-mae:1104.73837
[17]	validation_0-mae:1084.06329
[18]	validation_0-mae:1064.20425
[19]	validation_0-mae:1048.05529
[20]	validation_0-mae:1031.67897
[21]	validation_0-mae:1016.95860
[22]	validation_0-mae:1003.42672
[23]	validation_0-mae:991.27308
[24]	validation_0-mae:978.36568
[25]	validation_0-mae:965.99918
[26]	validation_0-mae:953.40058
[27]	validation_0-mae:941.89158
[28]	validation_0-mae:938.42138
[29]	validation_0-mae:935.11103
[30]	validation_0-mae:925.1

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=7, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, objective='reg:absoluteerror', ...)

## 6- Evaluation

Prediction

In [14]:
xgb_pred = my_xgb.predict(X_test)

Error

In [15]:
xgb_mae = mean_absolute_error(test_data[['generation_W', 'load_W']], xgb_pred)

In [16]:
print(f"MAE for PV Generation and Load: {xgb_mae:.4f}")

MAE for PV Generation and Load: 1178.7056
